# RL Run Analysis

This notebook analyzes W&B runs from RL training experiments (using `hf_trainer` with GRPOTrainer).
Fetches and visualizes reward metrics, parsing statistics, training progress, and sample outputs.

**Setup:** Ensure wandb is configured (`wandb login` or API key in `.env`).

In [ ]:
import html
import json
import typing

import IPython.display as ipy_display
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pyine.utils.filesystem
import pyine.utils.notebooks as nb_utils
import pyine.utils.reprod
import pyine.utils.wandb_utils as wandb_utils

In [ ]:
pyine.utils.reprod.entrypoint_setup()
nb_utils.setup_notebook_plotting(use_seaborn=True, seaborn_style="whitegrid")

# ------------ CONFIGURATION ------------
WANDB_PROJECT = "pyine-tests"  # change to your project
WANDB_ENTITY = None  # optional: your team/user entity

# option 1: specify run directly by URL or ID (takes priority)
WANDB_RUN_URL = ""  # e.g., "https://wandb.ai/entity/project/runs/abc123"
WANDB_RUN_ID = ""  # e.g., "abc123"

# option 2: search for runs using filters (used when URL/ID not specified)
# W&B API filters (applied server-side) - use MongoDB-style query syntax
RUN_FILTERS: dict[str, typing.Any] = {
    # RL runs have num_generations > 0 (GRPO generates multiple completions per prompt)
    "config.num_generations": {"$gt": 0},
    # "state": "finished",  # uncomment to only show completed runs
    # "group": "...",  # uncomment to filter by run group
}
# optional client-side filter (set to None to skip); applied after server-side filters
RUN_FILTER_FN: typing.Callable[[typing.Any], bool] | None = None
SELECTED_RUN_IDX = 0  # which run to select from search results (0 = most recent)

# optional: parquet cache path for large runs (set to None to disable)
history_cache_dir = pyine.utils.filesystem.get_data_cache_subdir("notebooks", "cached_wandb_runs")

In [ ]:
# load run: try direct URL/ID first, then fall back to filter-based search
if WANDB_RUN_URL or WANDB_RUN_ID:
    # option 1: direct run specification
    run = wandb_utils.get_wandb_run(WANDB_RUN_URL, WANDB_RUN_ID, WANDB_PROJECT, WANDB_ENTITY)
    print(f"Loaded run directly: {run.name} ({run.url})")
else:
    # option 2: search for runs using filters
    print(f"Searching for runs with filters: {RUN_FILTERS}")
    runs = wandb_utils.fetch_runs(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        filters=RUN_FILTERS if RUN_FILTERS else None,
        order="-created_at",  # newest first
        per_page=50,
    )
    # apply client-side filter if provided (e.g., to filter for RL runs)
    if RUN_FILTER_FN is not None:
        runs = [r for r in runs if RUN_FILTER_FN(r)]
        print(f"After client-side filter: {len(runs)} RL runs")
    if not runs:
        raise ValueError(f"No runs found matching filters (server: {RUN_FILTERS}, client: {RUN_FILTER_FN})")
    print(f"Found {len(runs)} matching runs:")
    for idx, r in enumerate(runs[:10]):  # show up to 10
        marker = " <-- selected" if idx == SELECTED_RUN_IDX else ""
        print(f"  [{idx}] {r.group}/{r.name} ({r.state}) - {r.created_at}{marker}")
    if len(runs) > 10:
        print(f"  ... and {len(runs) - 10} more")
    if len(runs) <= SELECTED_RUN_IDX:
        raise ValueError(f"SELECTED_RUN_IDX={SELECTED_RUN_IDX} but only {len(runs)} runs found")
    run = runs[SELECTED_RUN_IDX]
    print(f"\nSelected run: {run.name} ({run.url})")

print(f"State: {run.state}, Created: {run.created_at}")

In [ ]:
# display key summary metrics and configuration
summary = dict(run.summary)
config = dict(run.config)

print("Run Configuration:")
config_keys_of_interest = [
    "model_name",
    "base_model",
    "dataset",
    "num_train_epochs",
    "per_device_train_batch_size",
    "learning_rate",
    "num_generations",
]
for key in config_keys_of_interest:
    if key in config:
        print(f"  {key}: {config[key]}")

# show nested config keys if present
if "grpo_config" in config:
    print("  grpo_config:")
    grpo = config["grpo_config"]
    for subkey in ["num_generations", "max_completion_length", "temperature", "beta"]:
        if subkey in grpo:
            print(f"    {subkey}: {grpo[subkey]}")

print("\nRun Summary (key metrics):")
summary_keys_of_interest = [
    "train/reward/run/total/mean",
    "train/reward/run/total/sample_count",
    "train/global_step",
    "_step",
]
for key in summary_keys_of_interest:
    if key in summary:
        print(f"  {key}: {summary[key]}")

In [ ]:
available_keys = wandb_utils.discover_metric_keys(run)

print("Available Metrics in This Run:")
for category, keys in available_keys.items():
    if keys:
        print(f"\n  {category} ({len(keys)} keys):")
        for key in keys:
            print(f"    - {key}")
    else:
        print(f"\n  {category}: (none)")

In [ ]:
# fetch important metrics
important_parsing_suffixes = [
    "/has_reasoning",
    "/has_answer",
    "/is_malformed",
    "_length_tokens",
]
important_parsing_keys = [
    key for key in available_keys["parsing"] if any(key.endswith(s) for s in important_parsing_suffixes)
]
important_trl_parts = [
    "entropy",
    "kl",
    "clip_ratio",
]
important_trl_keys = [key for key in available_keys["trl"] if any(s in key for s in important_trl_parts)]
important_other_suffixes = [
    "/failure_ratio",
    "/step_time",
    "/steps_per_second",
    "/samples_per_second",
]
important_other_keys = [
    key for key in available_keys["other"] if any(key.endswith(s) for s in important_other_suffixes)
]
important_keys = (
    available_keys["reward_total"]
    + available_keys["reward_terms"]
    + available_keys["step_keys"]
    + important_trl_keys
    + important_parsing_keys
    + important_other_keys
)

# only cache history for finished runs (incomplete runs would have stale cached data)
history_cache_path = history_cache_dir / f"{run.id}.parquet" if run.state != "running" else None
if history_cache_path is None:
    print(f"Run state is '{run.state}' - caching disabled (will fetch fresh data)")
history_df = wandb_utils.fetch_history_df(
    run,
    keys=important_keys if important_keys else None,
    cache_path=history_cache_path,
)
step_key = wandb_utils.resolve_step_key(history_df)
time_key = wandb_utils.resolve_time_key(history_df)

print(f"\nFetched {len(history_df)} history rows with {len(history_df.columns)} columns")
print(f"Using step key: {step_key}")
if time_key:
    print(f"Time key available: {time_key}")

In [ ]:
history_df  # noqa: B018 (for display purposes)

## Reward Metrics

In [ ]:
def _compute_rolling_reward_stats(
    df: pd.DataFrame,
    step_key: str,
    reward_key: str,
    window_frac: float = 0.1,
    min_samples: int = 10,
    ci_local_frac: float = 0.05,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Computes rolling mean reward and 95% CI over training steps.

    Args:
        df: DataFrame with step and reward columns.
        step_key: Name of the step column (x-axis).
        reward_key: Name of the reward column (y-axis).
        window_frac: Fraction of data for rolling window (0.0-1.0). Larger = smoother.
        min_samples: Minimum samples in window to compute statistics.
        ci_local_frac: Fraction of x-range for local density estimation.

    Returns:
        Tuple of (x_values, rolling_mean, lower_ci, upper_ci). Empty arrays if insufficient data.
    """
    valid_df = df[[step_key, reward_key]].dropna().copy()
    if len(valid_df) < min_samples:
        return np.array([]), np.array([]), np.array([]), np.array([])
    valid_df = valid_df.sort_values(step_key).reset_index(drop=True)
    window_size = max(min_samples, int(len(valid_df) * window_frac))
    reward_series = valid_df[reward_key]
    rolling = reward_series.rolling(window=window_size, center=True, min_periods=min_samples)
    rolling_mean = rolling.mean()
    rolling_std = rolling.std()
    valid_mask = rolling_mean.notna()
    x_vals = valid_df.loc[valid_mask, step_key].values
    mean_vals = rolling_mean[valid_mask].values
    std_vals = rolling_std[valid_mask].values
    if len(x_vals) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([])
    unique_x, unique_idx = np.unique(x_vals, return_index=True)
    mean_out = mean_vals[unique_idx]
    std_out = std_vals[unique_idx]
    # compute local density for CI calculation
    x_range = x_vals.max() - x_vals.min()
    if x_range > 0:
        half_width = x_range * ci_local_frac / 2
        local_counts = np.array([np.sum((x_vals >= x - half_width) & (x_vals <= x + half_width)) for x in unique_x])
        local_counts = np.maximum(local_counts, 1)
    else:
        local_counts = np.full(len(unique_x), len(x_vals))
    standard_error = std_out / np.sqrt(local_counts)
    # z-score for 95% confidence interval (two-tailed: 2.5% in each tail)
    z_score_95_ci = 1.96  # this is the 97.5th percentile of the standard normal distribution
    lower_ci = mean_out - z_score_95_ci * standard_error
    upper_ci = mean_out + z_score_95_ci * standard_error
    return unique_x, mean_out, lower_ci, upper_ci


def plot_reward_over_time(
    history_df: pd.DataFrame,
    reward_key: str = "train/reward/total",
    step_key: str = "train/global_step",
    ax: plt.Axes | None = None,
    figsize: tuple[int, int] = (12, 5),
    title: str | None = None,
    window_frac: float = 0.15,
    show_scatter: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot reward metric over training steps with rolling average and 95% CI.

    Args:
        history_df: DataFrame with history data.
        reward_key: Name of the reward column to plot.
        step_key: Name of the step column (x-axis).
        ax: Optional matplotlib axes to plot on.
        figsize: Figure size if creating new figure.
        title: Chart title (auto-generated if None).
        window_frac: Fraction of data for rolling window (0.0-1.0).
        show_scatter: Whether to show individual data points.
        show_legend: Whether to show the legend.
        x_percentile_range: Percentile range for x-axis limits, e.g. (1, 99).

    Returns:
        The matplotlib Figure object.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()
    if reward_key not in history_df.columns:
        ax.text(0.5, 0.5, f"Reward key '{reward_key}' not found", ha="center", va="center", transform=ax.transAxes)
        return fig
    valid_df = history_df[[step_key, reward_key]].dropna()
    if len(valid_df) == 0:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center", transform=ax.transAxes)
        return fig
    # compute percentile-based x-axis limits
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None and len(valid_df) > 0:
        x_values = valid_df[step_key].values
        x_low = float(np.percentile(x_values, x_percentile_range[0]))
        x_high = float(np.percentile(x_values, x_percentile_range[1]))
        if x_low < x_high:
            padding = (x_high - x_low) * 0.05
            x_limits = (x_low - padding, x_high + padding)
    # compute rolling statistics
    x_curve, mean_curve, lower_ci, upper_ci = _compute_rolling_reward_stats(
        history_df, step_key, reward_key, window_frac=window_frac
    )
    if len(x_curve) == 0:
        ax.text(0.5, 0.5, "Insufficient data for curve", ha="center", va="center", transform=ax.transAxes)
        return fig
    # plot scatter points
    if show_scatter:
        ax.scatter(
            valid_df[step_key],
            valid_df[reward_key],
            alpha=0.2,
            s=8,
            color="#2C7BB6",
            label=f"Raw values (n={len(valid_df)})",
        )
    # plot CI band and rolling mean
    ax.fill_between(x_curve, lower_ci, upper_ci, alpha=0.25, color="#2C7BB6", label="95% CI")
    ax.plot(x_curve, mean_curve, color="#2C7BB6", linewidth=2, label="Rolling mean")
    # configure axes
    ax.set_xlabel(step_key)
    ax.set_ylabel(reward_key)
    ax.set_title(title or f"Reward Over Training ({reward_key})")
    if x_limits is not None:
        ax.set_xlim(*x_limits)
    ax.grid(axis="y", alpha=0.3)
    if show_legend:
        ax.legend(loc="upper left", fontsize=8)
    return fig


fig = plot_reward_over_time(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

In [ ]:
# color palette for multi-line plots (colorblind-friendly)
TERM_COLORS = ["#2C7BB6", "#D7191C", "#1A9641", "#FDAE61", "#9970AB", "#E7298A", "#66A61E", "#E6AB02"]


def plot_reward_terms_over_time(
    history_df: pd.DataFrame,
    term_keys: list[str],
    step_key: str = "_step",
    ax: plt.Axes | None = None,
    figsize: tuple[int, int] = (14, 6),
    title: str | None = None,
    window_frac: float = 0.15,
    show_ci: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot all reward terms over time on the same axes with rolling mean and 95% CI.

    Args:
        history_df: DataFrame with history data.
        term_keys: List of reward term column names to plot.
        step_key: Name of the step column (x-axis).
        ax: Optional matplotlib axes to plot on.
        figsize: Figure size if creating new figure.
        title: Chart title (auto-generated if None).
        window_frac: Fraction of data for rolling window (0.0-1.0).
        show_ci: Whether to show 95% CI bands.
        show_legend: Whether to show the legend.
        x_percentile_range: Percentile range for x-axis limits, e.g. (1, 99).

    Returns:
        The matplotlib Figure object.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()
    if not term_keys:
        ax.text(0.5, 0.5, "No reward term keys found", ha="center", va="center", transform=ax.transAxes)
        return fig
    # compute global x-axis limits from all terms
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None:
        all_x_values = []
        for key in term_keys:
            if key in history_df.columns:
                valid_df = history_df[[step_key, key]].dropna()
                if len(valid_df) > 0:
                    all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
            x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.05
                x_limits = (x_low - padding, x_high + padding)
    # plot each term
    plotted_count = 0
    for key_idx, key in enumerate(term_keys):
        if key not in history_df.columns:
            continue
        valid_df = history_df[[step_key, key]].dropna()
        if len(valid_df) == 0:
            continue
        color = TERM_COLORS[key_idx % len(TERM_COLORS)]
        # compute rolling statistics
        x_curve, mean_curve, lower_ci, upper_ci = _compute_rolling_reward_stats(
            history_df, step_key, key, window_frac=window_frac
        )
        if len(x_curve) == 0:
            continue
        # plot CI band (if enabled)
        if show_ci:
            ax.fill_between(x_curve, lower_ci, upper_ci, alpha=0.15, color=color)
        # plot rolling mean
        n_samples = len(valid_df)
        ax.plot(x_curve, mean_curve, color=color, linewidth=2, label=f"{key} (n={n_samples})")
        plotted_count += 1
    if plotted_count == 0:
        ax.text(0.5, 0.5, "No valid data for any term", ha="center", va="center", transform=ax.transAxes)
        return fig
    # configure axes
    ax.set_xlabel(step_key)
    ax.set_ylabel("reward value")
    ax.set_title(title or "Reward Terms Over Training")
    if x_limits is not None:
        ax.set_xlim(*x_limits)
    ax.grid(axis="y", alpha=0.3)
    if show_legend:
        ax.legend(loc="best", fontsize=8)
    return fig


fig = plot_reward_terms_over_time(history_df, available_keys["reward_terms"], step_key=step_key)
plt.tight_layout()
plt.show()

## Parsing Statistics

In [ ]:
def plot_parsing_stats(
    history_df: pd.DataFrame,
    parsing_keys: list[str],
    step_key: str = "_step",
    figsize: tuple[int, int] = (14, 10),
    window_frac: float = 0.15,
    show_ci: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot parsing-related metrics over time with rolling mean and 95% CI.

    Args:
        history_df: DataFrame with history data.
        parsing_keys: List of parsing metric column names to plot.
        step_key: Name of the step column (x-axis).
        figsize: Figure size.
        window_frac: Fraction of data for rolling window (0.0-1.0).
        show_ci: Whether to show 95% CI bands.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    # group keys by type
    length_keys = [k for k in parsing_keys if "length" in k]
    ratio_keys = [k for k in parsing_keys if "has_" in k or "missing" in k or "malformed" in k]
    groups = [(length_keys, "Output Lengths", "length"), (ratio_keys, "Parsing Ratios", "ratio")]
    groups = [(keys, title, ylabel) for keys, title, ylabel in groups if keys]
    if not groups:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, "No parsing metrics found", ha="center", va="center", transform=ax.transAxes)
        return fig
    fig, axes = plt.subplots(len(groups), 1, figsize=figsize)
    if len(groups) == 1:
        axes = [axes]
    # compute global x-axis limits
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None:
        all_x_values = []
        for key in parsing_keys:
            if key in history_df.columns:
                valid_df = history_df[[step_key, key]].dropna()
                if len(valid_df) > 0:
                    all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
            x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.05
                x_limits = (x_low - padding, x_high + padding)
    # plot each group
    for ax_idx, (keys, title, ylabel) in enumerate(groups):
        ax = axes[ax_idx]
        plotted_count = 0
        for key_idx, key in enumerate(keys[:6]):
            if key not in history_df.columns:
                continue
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) == 0:
                continue
            color = TERM_COLORS[key_idx % len(TERM_COLORS)]
            x_curve, mean_curve, lower_ci, upper_ci = _compute_rolling_reward_stats(
                history_df, step_key, key, window_frac=window_frac
            )
            if len(x_curve) == 0:
                continue
            if show_ci:
                ax.fill_between(x_curve, lower_ci, upper_ci, alpha=0.15, color=color)
            n_samples = len(valid_df)
            ax.plot(x_curve, mean_curve, color=color, linewidth=2, label=f"{key} (n={n_samples})")
            plotted_count += 1
        if plotted_count == 0:
            ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel(step_key)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if x_limits is not None:
            ax.set_xlim(*x_limits)
        ax.grid(axis="y", alpha=0.3)
        if show_legend and plotted_count > 0:
            ax.legend(loc="best", fontsize=8)
    return fig


# use parsing keys that exist in history_df (not all discovered keys)
actual_parsing_keys = [k for k in history_df.columns if "/parsing/" in k]
fig = plot_parsing_stats(history_df, actual_parsing_keys, step_key=step_key)
plt.tight_layout()
plt.show()

## Sample Output Viewer

In [ ]:
def normalize_rewards_table(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize rewards table to consistent schema."""
    df = df.copy()
    json_columns = [
        "reward_terms_json",
        "reward_terms_raw_json",
        "reward_metrics_json",
        "categories_json",
        "tags_json",
    ]
    for col in json_columns:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    required_cols = [
        "sample_id",
        "step",
        "prompt",
        "expected_output",
        "model_output",
        "reward_total",
        "generation_idx",
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = None
    # use _logged_step as the authoritative step if available
    if "_logged_step" in df.columns:
        df["step"] = df["_logged_step"].fillna(df["step"])
    # sort by step numerically (ensure numeric type for proper sorting)
    if "step" in df.columns:
        df["step"] = pd.to_numeric(df["step"], errors="coerce")
        df = df.sort_values("step", na_position="last").reset_index(drop=True)
    return df


# fetch tables with their true logged steps from W&B
rewards_table = wandb_utils.fetch_tables_with_steps(run, "train/generation_details")
if rewards_table is not None:
    rewards_table = normalize_rewards_table(rewards_table)
    print(f"Loaded {len(rewards_table)} sample records from rewards table")
    print(f"Columns: {list(rewards_table.columns)}")
    if "_logged_step" in rewards_table.columns:
        unique_steps = rewards_table["_logged_step"].nunique()
        print(f"Samples from {unique_steps} unique logged steps")
else:
    print("No rewards table found (log_tables may be disabled in reward logging config)")

In [ ]:
def get_parsing_metric(row: pd.Series, key: str, default: int = 0) -> int:
    """Get a parsing metric from reward_metrics_json (handles prefixed keys)."""
    metrics = row.get("reward_metrics_json")
    if isinstance(metrics, dict):
        # try exact key first, then search for keys ending with the expected suffix
        if key in metrics:
            return metrics[key]
        suffix = f"parsing/{key}"
        for k, v in metrics.items():
            if k.endswith(suffix):
                return v
    return default


def display_sample(row: pd.Series) -> None:
    """Display a single sample from the rewards table."""
    # check for missing fields using actual values
    reasoning = row.get("reasoning")
    has_reasoning = bool(reasoning) if isinstance(reasoning, str) else False
    has_answer = get_parsing_metric(row, "has_answer", default=1) == 1
    # build status indicators
    status_parts = []
    if not has_reasoning:
        status_parts.append('<span style="color: orange; font-weight: bold;">⚠ NO REASONING</span>')
    if not has_answer:
        status_parts.append('<span style="color: orange; font-weight: bold;">⚠ NO ANSWER</span>')
    status_html = " ".join(status_parts) if status_parts else '<span style="color: green;">✓ OK</span>'
    html_parts = []
    html_parts.append(f"<h4>Sample: {row.get('sample_id', 'N/A')} (Step {row.get('step', 'N/A')}) {status_html}</h4>")
    html_parts.append(f"<b>Reward:</b> {row.get('reward_total', 'N/A'):.4f}<br>")
    # generation index
    gen_idx = row.get("generation_idx")
    if gen_idx is not None:
        html_parts.append(f"<b>Generation Index:</b> {gen_idx}<br>")
    # categories
    cats = row.get("categories_json", [])
    if cats:
        html_parts.append(f"<b>Categories:</b> {', '.join(cats) if isinstance(cats, list) else str(cats)}<br>")
    # reward terms
    terms = row.get("reward_terms_json", {})
    if terms and isinstance(terms, dict):
        terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in terms.items())
        html_parts.append(f"<b>Terms:</b> {terms_str}<br>")
    # raw reward terms (if available)
    raw_terms = row.get("reward_terms_raw_json", {})
    if raw_terms and isinstance(raw_terms, dict):
        raw_terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in raw_terms.items())
        html_parts.append(f"<b>Raw Terms:</b> {raw_terms_str}<br>")
    html_parts.append("<hr>")
    # prompt (escape HTML to show raw tags like <final>)
    prompt = row.get("prompt") or ""
    html_parts.append(f"<b>Prompt:</b><br><pre>{html.escape(prompt)}</pre>")
    # expected output
    expected = row.get("expected_output")
    if expected:
        html_parts.append(f"<b>Expected Output:</b><br><pre>{html.escape(expected)}</pre>")
    # model output (escape HTML to show raw tags like <final>)
    output = row.get("model_output") or ""
    html_parts.append(f"<b>Model Output:</b><br><pre>{html.escape(output)}</pre>")
    # reasoning
    if has_reasoning:
        html_parts.append(f"<b>Reasoning:</b><br><pre>{html.escape(reasoning)}</pre>")
    else:
        html_parts.append('<b>Reasoning:</b> <span style="color: orange;">(missing)</span><br>')
    # final answer
    answer = row.get("final_answer")
    if answer:
        html_parts.append(f"<b>Final Answer:</b><br><pre>{html.escape(answer)}</pre>")
    elif not has_answer:
        html_parts.append('<b>Final Answer:</b> <span style="color: orange;">(missing)</span><br>')
    ipy_display.display(ipy_display.HTML("".join(html_parts)))


if rewards_table is not None and len(rewards_table) > 0:
    print("Displaying first sample:")
    display_sample(rewards_table.iloc[0])
else:
    print("No samples to display")

In [ ]:
# interactive sample browser using ipywidgets
try:
    import ipywidgets

    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available - interactive browser disabled")


def create_sample_browser(
    rewards_table: pd.DataFrame,
    run_url: str,
) -> "ipywidgets.VBox":
    """Create an interactive sample browser widget."""
    current_df = rewards_table.copy()
    # filter controls
    show_missing_answer = ipywidgets.Checkbox(value=False, description="Missing answer")
    show_missing_reasoning = ipywidgets.Checkbox(value=False, description="Missing reasoning")
    # navigation
    idx_slider = ipywidgets.IntSlider(value=0, min=0, max=max(0, len(current_df) - 1), description="Index:")
    prev_btn = ipywidgets.Button(description="< Prev")
    next_btn = ipywidgets.Button(description="Next >")
    # output area
    output = ipywidgets.Output()
    info_label = ipywidgets.HTML(value=f"Total samples: {len(current_df)}")

    def apply_filters() -> None:
        nonlocal current_df
        df = rewards_table.copy()
        if show_missing_answer.value:
            df = df[df.apply(lambda r: get_parsing_metric(r, "has_answer", default=1) == 0, axis=1)]
        if show_missing_reasoning.value:
            df = df[df["reasoning"].apply(lambda x: not x if isinstance(x, str) else True)]
        current_df = df
        idx_slider.max = max(0, len(current_df) - 1)
        idx_slider.value = min(idx_slider.value, idx_slider.max)
        info_label.value = f"Showing {len(current_df)} of {len(rewards_table)} samples"
        update_display(None)

    def update_display(_: typing.Any) -> None:
        output.clear_output()
        with output:
            if len(current_df) == 0:
                print("No samples match current filters")
                return
            idx = idx_slider.value
            if idx < len(current_df):
                display_sample(current_df.iloc[idx])

    def on_prev(_: typing.Any) -> None:
        if idx_slider.value > 0:
            idx_slider.value -= 1

    def on_next(_: typing.Any) -> None:
        if idx_slider.value < idx_slider.max:
            idx_slider.value += 1

    # connect handlers
    idx_slider.observe(update_display, names="value")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    show_missing_answer.observe(lambda _: apply_filters(), names="value")
    show_missing_reasoning.observe(lambda _: apply_filters(), names="value")
    # initial display
    update_display(None)
    # layout
    filter_box = ipywidgets.HBox([show_missing_answer, show_missing_reasoning])
    nav_box = ipywidgets.HBox([prev_btn, idx_slider, next_btn])
    url_html = ipywidgets.HTML(value=f"<a href='{run_url}' target='_blank'>Open in W&B</a>")
    return ipywidgets.VBox([info_label, filter_box, nav_box, url_html, output])


if WIDGETS_AVAILABLE and rewards_table is not None and len(rewards_table) > 0:
    browser = create_sample_browser(rewards_table, run.url)
    ipy_display.display(browser)
elif rewards_table is None:
    print("No rewards table available for browsing")
else:
    print("No samples in rewards table")

In [ ]:
# sample filtering configuration
FILTER_STEP_RANGE = (None, None)  # (min_step, max_step) or None
FILTER_REWARD_RANGE = (None, None)  # (min_reward, max_reward) or None
FILTER_CATEGORIES: list[str] = []  # list of category strings to include (empty = all)


def filter_samples(
    rewards_table: pd.DataFrame,
    step_range: tuple[int | None, int | None] = (None, None),
    reward_range: tuple[float | None, float | None] = (None, None),
    categories: list[str] | None = None,
) -> pd.DataFrame:
    """Filter samples by step, reward, or category."""
    df = rewards_table.copy()
    # step range filter
    if step_range[0] is not None and "step" in df.columns:
        df = df[df["step"] >= step_range[0]]
    if step_range[1] is not None and "step" in df.columns:
        df = df[df["step"] <= step_range[1]]
    # reward range filter
    if reward_range[0] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] >= reward_range[0]]
    if reward_range[1] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] <= reward_range[1]]
    # category filter
    if categories and "categories_json" in df.columns:

        def has_category(cats: typing.Any) -> bool:
            if not isinstance(cats, list):
                return False
            return any(c in cats for c in categories)

        df = df[df["categories_json"].apply(has_category)]
    return df


if rewards_table is not None:
    filtered = filter_samples(rewards_table, FILTER_STEP_RANGE, FILTER_REWARD_RANGE, FILTER_CATEGORIES)
    print(f"Filtered to {len(filtered)} samples (from {len(rewards_table)} total)")
else:
    filtered = None
    print("No rewards table to filter")

In [ ]:
def plot_reward_distributions(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 10),
) -> matplotlib.figure.Figure:
    """Plot reward distributions."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    # 1. total reward histogram
    ax = axes[0, 0]
    if "reward_total" in rewards_table.columns:
        rewards = rewards_table["reward_total"].dropna()
        ax.hist(rewards, bins=50, color="#2C7BB6", alpha=0.7, edgecolor="black")
        ax.axvline(rewards.mean(), color="red", linestyle="--", label=f"Mean: {rewards.mean():.3f}")
        ax.set_xlabel("Reward")
        ax.set_ylabel("Count")
        ax.legend()
    ax.set_title("Total Reward Distribution")
    ax.grid(True, alpha=0.3)
    # 2. reward over steps
    ax = axes[0, 1]
    if "step" in rewards_table.columns and "reward_total" in rewards_table.columns:
        ax.scatter(rewards_table["step"], rewards_table["reward_total"], alpha=0.3, s=10)
    ax.set_title("Reward vs Step")
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    # 3. reward terms breakdown (if available)
    ax = axes[1, 0]
    if "reward_terms_json" in rewards_table.columns:
        # extract term values
        term_data: dict[str, list[float]] = {}
        for terms in rewards_table["reward_terms_json"].dropna():
            if isinstance(terms, dict):
                for term, value in terms.items():
                    if term not in term_data:
                        term_data[term] = []
                    term_data[term].append(value)
        if term_data:
            term_names = list(term_data.keys())
            term_values = [term_data[t] for t in term_names]
            ax.boxplot(term_values, labels=term_names)
            ax.set_xticklabels(term_names, rotation=45, ha="right")
    ax.set_title("Reward Terms Distribution")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    # 4. rewards by category (if available)
    ax = axes[1, 1]
    if "categories_json" in rewards_table.columns and "reward_total" in rewards_table.columns:
        # group by first category
        cat_rewards: dict[str, list[float]] = {}
        for _idx, row in rewards_table.iterrows():
            cats = row.get("categories_json", [])
            reward = row.get("reward_total")
            if isinstance(cats, list) and cats and reward is not None:
                cat = cats[0]  # use first category
                if cat not in cat_rewards:
                    cat_rewards[cat] = []
                cat_rewards[cat].append(reward)
        if cat_rewards:
            cat_names = list(cat_rewards.keys())[:8]  # limit to 8
            cat_values = [cat_rewards[c] for c in cat_names]
            ax.boxplot(cat_values, labels=cat_names)
            ax.set_xticklabels(cat_names, rotation=45, ha="right", fontsize=8)
    ax.set_title("Rewards by Category")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_reward_distributions(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for distribution analysis")

In [ ]:
def plot_length_vs_reward(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 5),
) -> matplotlib.figure.Figure:
    """Scatter plot: output/reasoning length vs reward."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    # compute lengths if not already present
    df = rewards_table.copy()
    if "output_length" not in df.columns and "model_output" in df.columns:
        df["output_length"] = df["model_output"].apply(lambda x: len(str(x)) if x else 0)
    if "reasoning_length" not in df.columns and "reasoning" in df.columns:
        df["reasoning_length"] = df["reasoning"].apply(lambda x: len(str(x)) if x else 0)
    # output length vs reward
    ax = axes[0]
    if "output_length" in df.columns and "reward_total" in df.columns:
        valid = df[["output_length", "reward_total"]].dropna()
        ax.scatter(valid["output_length"], valid["reward_total"], alpha=0.3, s=10, c="#2C7BB6")
        # add trend line
        if len(valid) > 10:
            z = np.polyfit(valid["output_length"], valid["reward_total"], 1)
            p = np.poly1d(z)
            x_line = np.linspace(valid["output_length"].min(), valid["output_length"].max(), 100)
            ax.plot(x_line, p(x_line), "r--", alpha=0.7, label="Trend")
            ax.legend()
    ax.set_xlabel("Output Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Output Length vs Reward")
    ax.grid(True, alpha=0.3)
    # reasoning length vs reward
    ax = axes[1]
    if "reasoning_length" in df.columns and "reward_total" in df.columns:
        valid = df[["reasoning_length", "reward_total"]].dropna()
        valid = valid[valid["reasoning_length"] > 0]  # only samples with reasoning
        if len(valid) > 0:
            ax.scatter(valid["reasoning_length"], valid["reward_total"], alpha=0.3, s=10, c="#7570B3")
            if len(valid) > 10:
                z = np.polyfit(valid["reasoning_length"], valid["reward_total"], 1)
                p = np.poly1d(z)
                x_line = np.linspace(valid["reasoning_length"].min(), valid["reasoning_length"].max(), 100)
                ax.plot(x_line, p(x_line), "r--", alpha=0.7, label="Trend")
                ax.legend()
        else:
            ax.text(
                0.5,
                0.5,
                "No samples with reasoning",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
    ax.set_xlabel("Reasoning Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Reasoning Length vs Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_length_vs_reward(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for length analysis")

## TRL Metrics

In [ ]:
def _find_metric_key(
    history_df: pd.DataFrame,
    base_name: str,
    prefixes: list[str] | None = None,
) -> str | None:
    """Find a metric key by checking bare name and common prefixes."""
    if prefixes is None:
        prefixes = ["", "train/", "eval/", "objective/", "policy/", "trl/"]
    # check exact match first
    if base_name in history_df.columns:
        return base_name
    # check with prefixes
    for prefix in prefixes:
        key = f"{prefix}{base_name}"
        if key in history_df.columns:
            return key
    return None


def plot_policy_stability(
    history_df: pd.DataFrame,
    step_key: str = "_step",
    figsize: tuple[int, int] = (14, 5),
    window_frac: float = 0.15,
    show_scatter: bool = True,
    show_ci: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = None,  # None = use full reward data range
) -> matplotlib.figure.Figure:
    """Plot KL divergence, entropy, and clip ratios over training.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        figsize: Figure size.
        window_frac: Fraction of data for rolling window.
        show_scatter: Whether to show individual data points.
        show_ci: Whether to show 95% CI bands.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits (None = use reward data range).

    Returns:
        The matplotlib Figure object.
    """
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    # find actual key names (may have prefixes)
    kl_key = _find_metric_key(history_df, "kl")
    entropy_key = _find_metric_key(history_df, "entropy")
    clip_keys = [k for k in history_df.columns if "clip_ratio" in k]
    # prefer train/ clip keys over eval/ for consistency
    train_clip_keys = [k for k in clip_keys if k.startswith("train/")]
    if train_clip_keys:
        clip_keys = train_clip_keys
    # compute x-axis limits from reward data (TRL metrics may have sparse coverage)
    x_limits: tuple[float, float] | None = None
    reward_keys = [k for k in history_df.columns if "/reward/" in k and "/total" in k]
    if reward_keys:
        all_x_values = []
        for key in reward_keys:
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) > 0:
                all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            if x_percentile_range is not None:
                x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
                x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            else:
                x_low = float(min(all_x_values))
                x_high = float(max(all_x_values))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.02
                x_limits = (x_low - padding, x_high + padding)

    def _plot_metric(ax: plt.Axes, key: str | None, title: str, ylabel: str, color: str) -> None:
        """Helper to plot a single metric with data availability info."""
        if key is None:
            ax.text(
                0.5,
                0.5,
                f"{ylabel} not found in history",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=10,
            )
            ax.set_title(title)
            ax.set_xlabel(step_key)
            ax.set_ylabel(ylabel)
            return
        valid_df = history_df[[step_key, key]].dropna()
        if len(valid_df) == 0:
            ax.text(
                0.5,
                0.5,
                f"{key} column exists but has no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=10,
            )
            ax.set_title(title)
            ax.set_xlabel(step_key)
            ax.set_ylabel(ylabel)
            return
        # show data range info
        step_min, step_max = valid_df[step_key].min(), valid_df[step_key].max()
        x_curve, mean_curve, lower_ci, upper_ci = _compute_rolling_reward_stats(
            history_df, step_key, key, window_frac=window_frac
        )
        if len(x_curve) > 0:
            if show_scatter:
                ax.scatter(
                    valid_df[step_key],
                    valid_df[key],
                    alpha=0.2,
                    s=5,
                    color=color,
                    label=f"Raw (n={len(valid_df)}, steps {step_min:.0f}-{step_max:.0f})",
                )
            if show_ci:
                ax.fill_between(x_curve, lower_ci, upper_ci, alpha=0.25, color=color)
            ax.plot(x_curve, mean_curve, color=color, linewidth=2, label="Rolling mean")
            if show_legend:
                ax.legend(loc="best", fontsize=7)
        else:
            ax.text(
                0.5, 0.5, f"Insufficient data (n={len(valid_df)})", ha="center", va="center", transform=ax.transAxes
            )
        ax.set_xlabel(step_key)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title}\n({key})" if key != ylabel else title)
        if x_limits is not None:
            ax.set_xlim(*x_limits)
        ax.grid(axis="y", alpha=0.3)

    # KL divergence
    _plot_metric(axes[0], kl_key, "KL Divergence (Policy vs Reference)", "kl", "#D7191C")
    # entropy
    _plot_metric(axes[1], entropy_key, "Policy Entropy (Exploration)", "entropy", "#1A9641")
    # clip ratio (multiple keys)
    ax = axes[2]
    if clip_keys:
        plotted_any = False
        for key_idx, key in enumerate(clip_keys[:4]):  # limit to 4 keys
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) == 0:
                continue
            color = TERM_COLORS[key_idx % len(TERM_COLORS)]
            x_curve, mean_curve, lower_ci, upper_ci = _compute_rolling_reward_stats(
                history_df, step_key, key, window_frac=window_frac
            )
            if len(x_curve) == 0:
                continue
            if show_ci:
                ax.fill_between(x_curve, lower_ci, upper_ci, alpha=0.15, color=color)
            step_min, step_max = valid_df[step_key].min(), valid_df[step_key].max()
            short_key = key.split("/")[-1] if "/" in key else key
            ax.plot(
                x_curve,
                mean_curve,
                color=color,
                linewidth=2,
                label=f"{short_key} (n={len(valid_df)}, {step_min:.0f}-{step_max:.0f})",
            )
            plotted_any = True
        if plotted_any and show_legend:
            ax.legend(loc="best", fontsize=7)
        if not plotted_any:
            ax.text(0.5, 0.5, "Clip ratio keys found but no data", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.text(0.5, 0.5, "Clip ratio not found in history", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel(step_key)
    ax.set_ylabel("clip_ratio")
    ax.set_title("Policy Update Clipping")
    if x_limits is not None:
        ax.set_xlim(*x_limits)
    ax.grid(axis="y", alpha=0.3)
    return fig


fig = plot_policy_stability(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

# print data availability summary
print("\nTRL Metrics Data Availability:")
kl_key = _find_metric_key(history_df, "kl")
entropy_key = _find_metric_key(history_df, "entropy")
clip_keys = [k for k in history_df.columns if "clip_ratio" in k]
for name, key in [("KL", kl_key), ("Entropy", entropy_key)]:
    if key:
        valid = history_df[key].dropna()
        if len(valid) > 0:
            steps = history_df.loc[valid.index, step_key]
            print(f"  {name} ({key}): {len(valid)} samples, steps {steps.min():.0f}-{steps.max():.0f}")
        else:
            print(f"  {name} ({key}): column exists but no data")
    else:
        print(f"  {name}: not found in history")
for key in clip_keys[:6]:
    valid = history_df[key].dropna()
    if len(valid) > 0:
        steps = history_df.loc[valid.index, step_key]
        print(f"  {key}: {len(valid)} samples, steps {steps.min():.0f}-{steps.max():.0f}")

## Training Speed & Other Metrics

In [ ]:
def plot_training_speed_metrics(
    history_df: pd.DataFrame,
    other_keys: list[str],
    step_key: str = "_step",
    time_key: str | None = None,
    figsize: tuple[int, int] = (14, 10),
    window_frac: float = 0.15,
    show_ci: bool = True,
    show_legend: bool = True,
) -> matplotlib.figure.Figure:
    """Plot training speed metrics and experiment progress over time.

    Args:
        history_df: DataFrame with history data.
        other_keys: List of "other" metric keys (failure_ratio, step_time, etc.).
        step_key: Name of the step column (x-axis).
        time_key: Name of the time column (_timestamp or _runtime). If None, auto-detected.
        figsize: Figure size.
        window_frac: Fraction of data for rolling window.
        show_ci: Whether to show 95% CI bands.
        show_legend: Whether to show legends.

    Returns:
        The matplotlib Figure object.
    """
    # group keys by metric type
    failure_keys = [k for k in other_keys if "failure" in k]
    step_time_keys = [k for k in other_keys if "step_time" in k]
    throughput_keys = [k for k in other_keys if "per_second" in k]
    # determine time key if not provided
    if time_key is None:
        time_key = "_runtime" if "_runtime" in history_df.columns else "_timestamp"
    has_time = time_key in history_df.columns
    # determine subplot layout: 2x2 if we have time data, otherwise 1x3
    if has_time:
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        axes = axes.flatten()
    else:
        fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    ax_idx = 0

    def _plot_group(
        ax: plt.Axes,
        keys: list[str],
        title: str,
        ylabel: str,
    ) -> None:
        """Plot a group of metrics on a single axis."""
        if not keys:
            ax.text(0.5, 0.5, f"No {ylabel} metrics found", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title)
            ax.set_xlabel(step_key)
            ax.set_ylabel(ylabel)
            return
        plotted_any = False
        for key_idx, key in enumerate(keys[:6]):
            if key not in history_df.columns:
                continue
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) == 0:
                continue
            color = TERM_COLORS[key_idx % len(TERM_COLORS)]
            x_curve, mean_curve, lower_ci, upper_ci = _compute_rolling_reward_stats(
                history_df, step_key, key, window_frac=window_frac
            )
            if len(x_curve) == 0:
                continue
            if show_ci:
                ax.fill_between(x_curve, lower_ci, upper_ci, alpha=0.15, color=color)
            short_key = key.split("/")[-1] if "/" in key else key
            ax.plot(x_curve, mean_curve, color=color, linewidth=2, label=f"{short_key} (n={len(valid_df)})")
            plotted_any = True
        if not plotted_any:
            ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel(step_key)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.3)
        if show_legend and plotted_any:
            ax.legend(loc="best", fontsize=8)

    # plot 1: failure ratios
    _plot_group(axes[ax_idx], failure_keys, "Failure Ratios Over Training", "ratio")
    ax_idx += 1
    # plot 2: step time
    _plot_group(axes[ax_idx], step_time_keys, "Step Time Over Training", "seconds")
    ax_idx += 1
    # plot 3: throughput (steps/samples per second)
    _plot_group(axes[ax_idx], throughput_keys, "Training Throughput", "per second")
    ax_idx += 1
    # plot 4: experiment speed (steps vs wall-clock time) - only if time data available
    if has_time and ax_idx < len(axes):
        ax = axes[ax_idx]
        valid_df = history_df[[step_key, time_key]].dropna().sort_values(time_key)
        if len(valid_df) > 0:
            time_vals = valid_df[time_key].values
            step_vals = valid_df[step_key].values
            # convert timestamp to relative time if needed
            if time_key == "_timestamp":
                time_vals = time_vals - time_vals[0]
                x_label = "Time (seconds since start)"
            else:
                x_label = "Runtime (seconds)"
            # convert to minutes if > 60 seconds
            if time_vals.max() > 60:
                time_vals = time_vals / 60
                x_label = x_label.replace("seconds", "minutes")
            # convert to hours if > 60 minutes
            if time_vals.max() > 60:
                time_vals = time_vals / 60
                x_label = x_label.replace("minutes", "hours")
            ax.plot(time_vals, step_vals, color="#2C7BB6", linewidth=2)
            ax.scatter(
                time_vals[:: max(1, len(time_vals) // 50)],
                step_vals[:: max(1, len(step_vals) // 50)],
                color="#2C7BB6",
                alpha=0.5,
                s=20,
            )
            # compute and show average speed
            if len(time_vals) > 1 and time_vals[-1] > time_vals[0]:
                total_steps = step_vals[-1] - step_vals[0]
                total_time = time_vals[-1] - time_vals[0]
                avg_speed = total_steps / total_time
                unit = x_label.split("(")[-1].split(")")[0].split()[-1]
                ax.text(
                    0.02,
                    0.98,
                    f"Avg: {avg_speed:.2f} steps/{unit}",
                    transform=ax.transAxes,
                    fontsize=9,
                    verticalalignment="top",
                    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
                )
            ax.set_xlabel(x_label)
            ax.set_ylabel(step_key)
            ax.set_title("Training Progress (Steps vs Wall-Clock Time)")
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, "No time data available", ha="center", va="center", transform=ax.transAxes)
            ax.set_title("Training Progress")
    return fig


# plot training speed metrics
fig = plot_training_speed_metrics(
    history_df,
    important_other_keys,
    step_key=step_key,
    time_key=time_key,
)
plt.tight_layout()
plt.show()

# print summary of available metrics
print("\nTraining Speed Metrics Summary:")
for key in important_other_keys:
    if key in history_df.columns:
        valid = history_df[key].dropna()
        if len(valid) > 0:
            print(f"  {key}: n={len(valid)}, mean={valid.mean():.4f}, std={valid.std():.4f}")
        else:
            print(f"  {key}: no data")
    else:
        print(f"  {key}: not in history")

# print time-based stats if available
if time_key and time_key in history_df.columns:
    valid_time = history_df[[step_key, time_key]].dropna()
    if len(valid_time) > 1:
        total_time = valid_time[time_key].max() - valid_time[time_key].min()
        total_steps = valid_time[step_key].max() - valid_time[step_key].min()
        print("\nExperiment Duration:")
        if total_time > 3600:
            print(f"  Total time: {total_time / 3600:.2f} hours")
        elif total_time > 60:
            print(f"  Total time: {total_time / 60:.2f} minutes")
        else:
            print(f"  Total time: {total_time:.2f} seconds")
        print(f"  Total steps: {total_steps:.0f}")
        print(f"  Average speed: {total_steps / total_time:.2f} steps/second")